In [1]:
import numpy as np
import sklearn as sk
import pandas as pd
import data_management as dm
import feature_builders as fb
import tree_models as tm
import model_wrappers as wp
from graph_helpers import plot_var_vs_loss, plot_ewma_bank

In [2]:
TICKERS = ["AAPL", "JPM", "XOM", "KO"]

WEIGHTS = {
    "AAPL": 0.25,
    "JPM": 0.25,
    "XOM": 0.25,
    "KO": 0.25,
}

df = pd.read_csv(
    "./data_general/4_ticker_data.csv",
    parse_dates=["Date"],
    index_col="Date",
)


tickers_data = df[TICKERS].pct_change().dropna()



In [3]:
calendar_last_180_days = dm.DailyGroundTruthCalendar(
    df, TICKERS, WEIGHTS,
    train_start="2024-06-03",   # Pazartesi, ~6 ay geriye (180 takvim günü civarı)
    train_end="2024-11-29",     # Cuma, Kasım'ın son iş günü
    forecast_start="2024-12-02",  # Pazartesi
    forecast_end="2024-12-31",
)

[WARNING] forecast_end=2024-12-31 falls on Tuesday, expected Friday (using it as-is).
[WARNING] [train] 4 weekday days had no data and were skipped (likely market holidays): 2024-06-19, 2024-07-04, 2024-09-02, 2024-11-28
[INFO] train set: 126 bars, 4 weekday days skipped
[WARNING] [forecast] 1 weekday day had no data and were skipped (likely market holidays): 2024-12-25
[INFO] forecast set: 21 bars, 1 weekday day skipped


In [23]:
training_feature_long, forecast_feature_long = fb.build_long_features(calendar_last_180_days, target_scale= 10000)

[INFO] long train: 504 rows -> 424 after dropna (80 dropped, 15.9%)
[INFO] long forecast: 88 rows -> 88 after dropna (0 dropped, 0.0%)


In [24]:
VAR_CONFIDENCE = 0.95

# Both banks are SINGLE-USE: training fills them in place and marks them trained, so they cannot be given to
# another training run (create new banks per run; creating them in this same cell keeps re-running it safe).
residual_bank = dm.ResidualBank(
    mu_mode="constant",                         # "constant": mu_hat = default_mu_hat | "rolling": mean of the last rolling_mu_hat_window_size returns
    default_mu_hat=0.0,
    rolling_mu_hat_window_size=50,              # only read when mu_mode="rolling"
)

# built from the calendar: tickers and weights are read from it (training checks it is the same calendar)
ewma_bank = dm.EwmaVarianceBank(
    calendar_last_180_days,
    ewma_lam=0.94,                              # RiskMetrics decay: larger = forgets slower
    ewma_min_periods=30,                        # the first N train bars seed the covariance (EWMA burn-in)
)

model = tm.xgboost_train(calendar_last_180_days, training_feature_long, residual_bank=residual_bank, ewma_bank=ewma_bank)

[INFO] usable_train_dates: 96/105 candidate dates survived burn-in (min_periods=29, train_data starts 2024-06-03, burn-in cutoff 2024-07-16, 9 candidate date(s) dropped).


In [25]:
# fills copies of the model's two banks with the forecast window (no VaR yet, nothing about reading involved)
residualbank, ewmabank = dm.calculate_multi_day_standart_residuals_from_long_format_model(
    pooled_model=model,
    calendar=calendar_last_180_days,
    long_forecast_feature=forecast_feature_long,
)

# reads the VaR off the filled bank, adds the realized loss, records the run under run_key.
# Everything about HOW to read is chosen here, so the same filled bank can be read again any other way.
var_df = calendar_last_180_days.var_from_residual_bank(
    residualbank,
    var_confidence=VAR_CONFIDENCE,                  # e.g. 0.95 -> takes the 5% lower quantile of the residuals
    multi_day_var_calculation_window_size=360,      # int: use at most the last N residuals; None: use all residuals available
    run_key="xgboost_long",                         # key in calendar.var_prediction_runs (reusing a key overwrites that run)
    multi_day_var_min_periods=30,                   # min prior residuals a day needs to get a VaR (else ValueError)
)

In [27]:
plot_var_vs_loss(var_df, residualbank)

In [19]:
# EwmaVarianceBank.portfolio_variance: her gunun portfoy varyansi (bank'in asil ciktisi, ResidualBank'a gider)
# Egitim (96 gun) + forecast (21 gun) = 117 satir; indeks = sonucun yasandigi gun
ewmabank.portfolio_variance.head()

2024-07-17    0.000043
2024-07-18    0.000056
2024-07-19    0.000032
2024-07-22    0.000031
2024-07-23    0.000030
Name: predicted_variance, dtype: float64

In [20]:
ewmabank.portfolio_variance.tail()

2024-12-24    0.000081
2024-12-26    0.000056
2024-12-27    0.000046
2024-12-30    0.000057
2024-12-31    0.000056
Name: predicted_variance, dtype: float64

In [21]:
ewmabank.cov_by_date

{Timestamp('2024-07-17 00:00:00'): array([[ 3.55360793e-04, -7.08067749e-05, -7.20516498e-05,
         -1.28072268e-05],
        [-7.08067749e-05,  1.45241611e-04,  5.40788403e-05,
         -1.47716984e-05],
        [-7.20516498e-05,  5.40788403e-05,  1.35864514e-04,
          1.51285535e-05],
        [-1.28072268e-05, -1.47716984e-05,  1.51285535e-05,
          5.26539067e-05]]),
 Timestamp('2024-07-18 00:00:00'): array([[ 3.72432068e-04, -8.96493001e-05, -8.86556698e-05,
         -3.42371273e-05],
        [-8.96493001e-05,  1.50414860e-04,  6.34204577e-05,
         -5.34492626e-07],
        [-8.86556698e-05,  6.34204577e-05,  1.39119545e-04,
          2.63206532e-05],
        [-3.42371273e-05, -5.34492626e-07,  2.63206532e-05,
          6.23294867e-05]]),
 Timestamp('2024-07-19 00:00:00'): array([[ 3.75387382e-04, -4.51262533e-05, -9.54857233e-05,
         -3.18050578e-05],
        [-4.51262533e-05,  2.01950628e-04,  4.08186427e-05,
          8.21441336e-08],
        [-9.54857233e-05

In [ ]:
# kac gunun kovaryansi kayitli ve son gunun matrisi (matrisler ticker etiketli DataFrame)
print(len(ewmabank.cov_by_date), "gun")
last_date = ewmabank.portfolio_variance.index[-1]
ewmabank.cov_by_date[last_date]

In [29]:
plot_ewma_bank(ewmabank)                                                  # varsayılan


In [13]:
plot_ewma_bank(ewmabank, split_date=calendar_last_180_days.train_data_last_index[0])   # eğitim/forecast sınırı


In [14]:
plot_ewma_bank(ewmabank, as_volatility=True)                              # sqrt ve % olarak


In [32]:
idx = residualbank.residuals.index
train_dates = idx[:len(model.residual_bank_in_sample.residuals)]   # bank'ın eğitim satırları (96)

usable = train_dates[30:]                                          # ilk 30 satırın önünde yeterli geçmiş yok
train_var = residualbank.multi_day_var(usable, VAR_CONFIDENCE, 60, 30)   # (tarihler, güven, pencere, min_periods)
train_var["loss"] = calendar_last_180_days.train_data.loc[train_var.index, "portfolio_loss"]
train_var

,varForecast,n_residuals_used,loss
Date,,,
2024-08-28,0.009652,30,0.002018
2024-08-29,0.009638,31,-0.009182
2024-08-30,0.010302,32,-0.003110
2024-09-03,0.0114,33,0.015179
2024-09-04,0.011494,34,0.007849
...,...,...,...
2024-11-22,0.008526,60,-0.005688
2024-11-25,0.007738,60,-0.003076
2024-11-26,0.007804,60,0.001476


In [33]:
plot_var_vs_loss(train_var, residualbank)